In [1]:
import sys
from pathlib import Path

# add src directory to python path
sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from logistic_regression import pre_process_tools

#### Phase 01 : Import data

In [2]:
pre_processed = pd.read_parquet('../src/data/pre_processed/features.parquet')

In [3]:
pre_processed.head(10)

,Elo_diff,Elo_mean,N_min,elo_surface_diff,elo_effective_diff,spec_diff,surface_exp_min,log_pts_diff,log_rank_diff,Surface_Clay,Surface_Grass,Surface_Hard,Best_of_5,Winner
0,0.000000,1500.000000,0.0,0.000000,0.000000,0.000000,0,NaN,0.585902,0.0,0.0,1.0,0,0
1,0.000000,1500.000000,0.0,0.000000,0.000000,0.000000,0,NaN,0.659246,0.0,0.0,1.0,0,0
2,0.000000,1500.000000,0.0,0.000000,0.000000,0.000000,0,NaN,-0.037041,0.0,0.0,1.0,0,1
3,0.000000,1500.000000,0.0,0.000000,0.000000,0.000000,0,NaN,0.843429,0.0,0.0,1.0,0,1
4,0.000000,1500.000000,0.0,0.000000,0.000000,0.000000,0,NaN,-1.749200,0.0,0.0,1.0,0,0
5,0.000000,1500.000000,0.0,0.000000,0.000000,0.000000,0,NaN,1.196251,0.0,0.0,1.0,0,0
6,-65.663195,1467.168402,0.0,-16.000000,-8.200000,-0.002000,0,NaN,2.151762,0.0,0.0,1.0,0,1
7,-77.927462,1461.036269,0.0,-16.377554,-8.393496,-0.002047,0,NaN,-2.095971,0.0,0.0,1.0,0,1
8,0.000000,1500.000000,0.0,0.000000,0.000000,0.000000,0,NaN,0.116724,0.0,0.0,1.0,0,0
9,-6.783409,1503.391704,0.0,0.402330,0.211223,0.000101,0,NaN,0.240054,0.0,0.0,1.0,0,0


#### Phase 02 : Imputation

Get rid of np.nan instances and drop one of dummies, because always one of them is a linear combination of others.

In [4]:
pre_processed = pre_process_tools.impute_nans(pre_processed)
pre_processed = pre_process_tools.drop_first_dummy(pre_processed)

In [5]:
pre_processed.shape

(52755, 13)

#### Phase 03 : Symmetric features

In [6]:
pre_processed = pre_process_tools.add_symmetric_features(pre_processed)

#### Phase 04 : Split train/test

In [8]:
y_name = 'Winner'
X_train, X_test, y_train, y_test = pre_process_tools.split_data(pre_processed, y_name)

In [9]:
print('X train shape:', X_train.shape)
print('X test  shape:', X_test .shape)
print('y train shape:', y_train.shape)
print('y test  shape:', y_test .shape)

X train shape: (42204, 15)
X test  shape: (10551, 15)
y train shape: (42204,)
y test  shape: (10551,)


#### Phase 05 : Scale

In [10]:
X_train

array([[  94.08071004, 1558.45301031,    3.        , ...,    0.        ,
           0.        ,   54.99300714],
       [ 223.80394472, 1630.82053593,   73.        , ...,    0.        ,
           0.        ,  292.78151992],
       [ 154.28205111, 1628.14912093,   98.        , ...,  154.28205111,
           0.        ,  197.71109225],
       ...,
       [ 189.96067239, 1974.51130905,  308.        , ...,  189.96067239,
           0.        ,  901.38487322],
       [ 196.7150655 , 1739.97710902,   47.        , ...,  196.7150655 ,
         196.7150655 ,  472.07112717],
       [ -89.58586511, 1679.62104517,   12.        , ...,  -89.58586511,
         -89.58586511, -160.91506723]], shape=(42204, 15))

Use sklearn's StandardScaler

In [11]:
X_train_scaled, X_test_scaled = pre_process_tools.scale_features(X_train, X_test)

In [12]:
X_train_scaled

array([[ 0.42409582, 10.66553366,  0.0222889 , ...,  0.        ,
         0.        ,  0.07654569],
       [ 1.00886056, 11.16079291,  0.5423633 , ...,  0.        ,
         0.        ,  0.40752754],
       [ 0.69547066, 11.14251064,  0.72810416, ...,  1.88491818,
         0.        ,  0.27519741],
       ...,
       [ 0.85630229, 13.51289816,  2.28832736, ...,  2.32081647,
         0.        ,  1.25465282],
       [ 0.88674965, 11.90782417,  0.34919281, ...,  2.40333727,
         1.67511015,  0.65708377],
       [-0.40383402, 11.49476736,  0.08915561, ..., -1.09450208,
        -0.7628607 , -0.2239804 ]], shape=(42204, 15))

#### Phase 06 : Train

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

base_model = LogisticRegression(
    penalty='l1',           # i tried elastic once and i got l1=1
    solver='liblinear',
    fit_intercept=False,    # important
    class_weight=None,      # data is already balanced by symmetrization
    max_iter=4000,          # seems segs need lots of iter
    tol=1e-4,
)

param_grid = {
    'C': np.logspace(-4, 2, 13),
    # 'l1_ratio': [0.0, 0.15, 0.3, 0.5, 0.7, 0.85, 1.0],
}

grid = GridSearchCV(
    base_model,
    param_grid,
    cv=TimeSeriesSplit(n_splits=5),
    scoring='neg_log_loss',
    n_jobs=-1,
    verbose=1,
    refit=True,
)

In [14]:
grid.fit(X_train_scaled, y_train)

print("Best params :", grid.best_params_)
print("CV LogLoss  :", -grid.best_score_)

best = grid.best_estimator_

Fitting 5 folds for each of 13 candidates, totalling 65 fits
Best params : {'C': np.float64(0.03162277660168379)}
CV LogLoss  : 0.589215494914915


In [15]:
w = pd.Series(best.coef_[0], index=pre_process_tools.FEATURES)

n_zero = (w == 0).sum()
print(f"Dropped features: {n_zero} / {len(w)}")
print("\nSurvivors by importance:")
print(w[w != 0].sort_values(key=abs, ascending=False))

Dropped features: 5 / 15

Survivors by importance:
Elo_diff            0.374911
log_pts_diff        0.346606
elo_x_bo5           0.171593
spec_diff           0.166177
elo_surface_diff    0.100310
elo_x_rel           0.099944
elo_x_field         0.079444
N_min               0.005755
elo_x_grass         0.002835
elo_x_clay         -0.001637
dtype: float64


#### Phase 07 : Evaluation

In [16]:
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss, accuracy_score

# Probabilities, not hard classes
proba = best.predict_proba(X_test_scaled)[:, 1]

print("Test LogLoss :", log_loss(y_test, proba))
print("Test AUC     :", roc_auc_score(y_test, proba))
print("Test Brier   :", brier_score_loss(y_test, proba))
print("Test Acc     :", accuracy_score(y_test, proba > 0.5))

Test LogLoss : 0.6163581329355997
Test AUC     : 0.7149927239147854
Test Brier   : 0.2145888970676116
Test Acc     : 0.6502701165766278
